In [ ]:
import matplotlib.pyplot as plt  # For plots
import seaborn  # For prettier plots
import time  # Deal with time stamps
import datetime  # Deal with time stamps again
import pandas  # Data stuff
import numpy as np  # For fitting data
import matplotlib.dates as mdates  # For handling dates
%matplotlib inline

# Strategies by date

In [ ]:
df = pandas.read_csv('data', header=None, names=['Dates', 'Strategies', 'Hash', 'Author'])
times = [time.strptime(time.asctime(time.gmtime(float(d)))) for d in df['Dates']]
dates = [datetime.datetime.fromtimestamp(time.mktime(t)) for t in times]
df['Dates'] = dates
df['AbsStrategies'] = [max(n, max(df['Strategies'][:i+1])) for i, n in enumerate(df['Strategies'])]

authors = {
"vince": "Vincent Knight",
"Vince Knight": "Vincent Knight",
"Vincent Knight": "Vincent Knight",
"Karol M. Langner": "Karol M. Langner",
"Jason Young": "Jason Young",
"jasonyoung": "Jason Young",
"Marc Harper":  "Marc Harper",
"Marc Harper, PhD": "Marc Harper",
"Owen Campbell": "Owen Campbell",
"geraint": "Geraint Palmer",
"slavinp": "Paul Slavin",
"slavinp": "Paul Slavin",
"Martin Chorley": "Martin Chorley",
"Martin": "Martin Jones",
"mojones": "Martin Jones",
"martin": "Martin Jones"}

df['Author'] = [authors.get(author, author) for author in df['Author']]

In [ ]:
x = mdates.date2num(list(df.Dates))
degree = 1
fit = np.polyfit(x, df.Strategies, degree)
line = np.poly1d(fit)
numerical_dates = np.linspace(x.min(), x.max())

projection = 50
project_x = np.linspace(numerical_dates[-1], numerical_dates[-1] + projection, projection)

plt.figure(figsize=(12, 8))

plt.step(df.Dates, df['Strategies'], where='post')
plt.plot(numerical_dates, line(numerical_dates), 'r')

plt.plot(project_x, line(project_x), 'r', linestyle='dashed', label='{0:.2f} strategies per day'.format(line[1]))
plt.xticks(rotation=70);

plt.ylabel('Number of strategies')
plt.legend(loc=2)
plt.savefig('actual_number_of_strategies.pdf');

The slope of the line:

In [ ]:
line[1]

This happens over

In [ ]:
int((df.Dates.iloc[-1] - df.Dates.iloc[0]).days)

days.

Here is the same analysis but ignoring a dip in the number of strategies (due to a commit that perhaps removed some strategies temporarily).

In [ ]:
x = mdates.date2num(list(df.Dates))
degree = 1
fit = np.polyfit(x, df.AbsStrategies, degree)
line = np.poly1d(fit)
numerical_dates = np.linspace(x.min(), x.max())

projection = 50
project_x = np.linspace(numerical_dates[-1], numerical_dates[-1] + projection, projection)

plt.figure(figsize=(12, 8))

plt.step(df.Dates, df['AbsStrategies'], where='post', label='Strategies')
plt.plot(numerical_dates, line(numerical_dates), 'r')

plt.plot(project_x, line(project_x), 'r', linestyle='dashed', label='{0:.2f} strategies per day'.format(line[1]))
plt.xticks(rotation=70);

plt.ylabel('Number of strategies')
plt.legend(loc=2)
plt.savefig('number_of_strategies.pdf');

Let us try and fit an ARIMA model.

In [ ]:
from statsmodels.tsa.arima_model import ARIMA

In [ ]:
order = (2, 1, 2)
model = ARIMA(df.Strategies.map(float), order, freq='D', dates=df['Dates'])
model = model.fit()

In [ ]:
x = mdates.date2num(list(df.Dates))
degree = 1
fit = np.polyfit(x, df.AbsStrategies, degree)
line = np.poly1d(fit)
numerical_dates = np.linspace(x.min(), x.max())

projection = 100
project_x = np.linspace(numerical_dates[-1], numerical_dates[-1] + projection, projection)

plt.figure(figsize=(12, 8))

plt.step(df.Dates, df['AbsStrategies'], where='post', label='Strategies')
#plt.plot(numerical_dates, line(numerical_dates), 'r')

plt.plot(project_x, model.forecast(projection)[0], 'r', linestyle='dashed', label='Arima projection: {}'.format(order))
plt.xticks(rotation=70);

plt.ylabel('Number of strategies')
plt.legend(loc=2)
plt.savefig('number_of_strategies_with_arima_projection.pdf');

Find contributions by author.

In [ ]:
df['Contributions'] = df.Strategies.diff()

In [ ]:
nbr_of_devs = 8

plt.figure(figsize=(12, 8))


ax = plt.subplot(111)
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width * 0.9, box.height])


devs = df['Author'].value_counts().index[:nbr_of_devs]
for author in devs:
    tmp_df = df[(df.Author == author)]

    try:
        plt.step(list(tmp_df.Dates), tmp_df.Contributions.cumsum(), label=author)

        plt.xticks(rotation=70)
    except KeyError:
        print(author)
        
ax.legend(loc='center left', bbox_to_anchor=(1, .5));

plt.savefig('contributions_{}_developers.pdf'.format(nbr_of_devs));

In [ ]:
tmp_df = df[(df.Author == "Martin Jones")]
plt.figure()
plt.step(list(tmp_df.Dates), tmp_df.Contributions.cumsum(), label=author)
plt.xticks(rotation=70);

In [ ]:
tmp_df